In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)


In [ ]:
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'BTC-USD'  # Any ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')
START_DATE = '2018-01-01'  # Any start date in YYYY-MM-DD format

# Download data from start date onwards
stock_data = yf.download(TICKER, start=START_DATE, interval='1d')

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")

# Display the downloaded data
stock_data


In [ ]:
# TECHNICAL ANALYSIS INDICATORS USING TA-LIB

# Make sure stock_data is available from the previous cell
if "stock_data" not in locals():
    raise ValueError("Please run the stock data download cell first")

# Extract OHLCV data (handling multi-level columns from yfinance)
if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[("Close", TICKER)].values
    high = stock_data[("High", TICKER)].values
    low = stock_data[("Low", TICKER)].values
    open_ = stock_data[("Open", TICKER)].values
    volume = stock_data[("Volume", TICKER)].values
else:
    close = stock_data["Close"].values
    high = stock_data["High"].values
    low = stock_data["Low"].values
    open_ = stock_data["Open"].values
    volume = stock_data["Volume"].values

print(f"Calculating technical indicators for {TICKER}...")

# Simple Moving Averages
sma_20 = talib.SMA(close, timeperiod=20)
sma_50 = talib.SMA(close, timeperiod=50)

# Exponential Moving Averages
ema_12 = talib.EMA(close, timeperiod=12)
ema_26 = talib.EMA(close, timeperiod=26)

# MACD
macd, macdsignal, macdhist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)

# RSI
rsi = talib.RSI(close, timeperiod=14)

# Stochastic RSI
stochrsi_k, stochrsi_d = talib.STOCHRSI(close, timeperiod=14, fastk_period=3, fastd_period=3, fastd_matype=0)

# VWAP (manual calculation)
typical_price = (high + low + close) / 3
price_volume = typical_price * volume
cumulative_price_volume = np.cumsum(price_volume)
cumulative_volume = np.cumsum(volume)
vwap = cumulative_price_volume / cumulative_volume

# Schaff Trend Cycle
cycle_period = 10
macd_cycle = talib.EMA(macd, timeperiod=cycle_period)
macd_smooth = talib.EMA(macd_cycle, timeperiod=cycle_period)
highest_macd = talib.MAX(macd_smooth, timeperiod=cycle_period)
lowest_macd = talib.MIN(macd_smooth, timeperiod=cycle_period)
stc_k = 100 * ((macd_smooth - lowest_macd) / (highest_macd - lowest_macd))
stc_d = talib.SMA(stc_k, timeperiod=3)

# Create indicators dataframe
indicators_df = pd.DataFrame({
    "Date": stock_data.index,
    "Close": close,
    "SMA_20": sma_20,
    "SMA_50": sma_50,
    "EMA_12": ema_12,
    "EMA_26": ema_26,
    "MACD": macd,
    "MACD_Signal": macdsignal,
    "MACD_Hist": macdhist,
    "RSI": rsi,
    "StochRSI_K": stochrsi_k,
    "StochRSI_D": stochrsi_d,
    "VWAP": vwap,
    "STC_K": stc_k,
    "STC_D": stc_d
})

print("All technical indicators calculated!")
print(f"Data shape: {indicators_df.shape}")
indicators_df.tail(5)

In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

# Expect stock_data and TICKER already exist
def select_close_series(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ('Close', ticker) in df.columns:
            s = df[('Close', ticker)]
        else:
            cols = [c for c in df.columns if 'Close' in str(c)]
            if not cols:
                raise KeyError("Close not found")
            s = df[cols[0]]
    else:
        s = df['Close']
    return s.astype(float).squeeze()

close = select_close_series(stock_data, TICKER)
close.name = 'price'

# Simple split
TRAIN_RATIO = 0.60 
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close   = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} \u2192 {train_close.index[-1].date()} | val={val_close.index[0].date()} \u2192 {val_close.index[-1].date()}")

RSI MEAN-REVERSION GRID SEARCH - TRAINING SET
----------------------------------------------

This section performs a comprehensive grid search optimization for the **RSI Mean-Reversion Strategy** using only the **training data**.

The goal is to find the optimal RSI Length / Oversold / Overbought combination that maximizes the Sharpe ratio on unseen data.

**Strategy Logic**: Buy when RSI crosses UP through the oversold threshold (mean-reversion entry). Sell when RSI crosses UP through the overbought threshold (profit-taking exit).

---

In [ ]:
# Define Parameter Ranges for RSI Mean-Reversion

# RSI parameters for mean-reversion strategy
rsi_len_range    = list(range(5, 61, 1))          # 56 values
oversold_range   = list(range(15, 46, 1))          # 31 values
overbought_range = list(range(55, 86, 1))          # 31 values

print("RSI Length Periods (lookback window):")
for i, period in enumerate(rsi_len_range, 1):
    print(f"  {i}. {period} periods")

print("Oversold Thresholds (entry trigger):")
for i, val in enumerate(oversold_range, 1):
    print(f"  {i}. {val}")

print("Overbought Thresholds (exit trigger):")
for i, val in enumerate(overbought_range, 1):
    print(f"  {i}. {val}")

# Generate all valid combinations: overbought > oversold + 10 (enforce minimum band gap)
rsi_combinations = []
for rsi_len in rsi_len_range:
    for oversold in oversold_range:
        for overbought in overbought_range:
            if overbought > oversold + 10:
                rsi_combinations.append((rsi_len, oversold, overbought))

print(f"Generated {len(rsi_combinations)} valid RSI Mean-Reversion combinations")
print("Note: Filter applied — overbought > oversold + 10 (minimum band gap)")
print("\n\U0001f4cb First 10 combinations preview:")
for i, (rsi_len, oversold, overbought) in enumerate(rsi_combinations[:10], 1):
    band_gap = overbought - oversold
    print(f"  {i:2d}. RSI_len: {rsi_len:2d} | Oversold: {oversold:2d} | Overbought: {overbought:2d} (gap={band_gap})")
if len(rsi_combinations) > 10:
    print(f"   ... and {len(rsi_combinations) - 10} more combinations")

print("\nReady to test all combinations on training data!")

In [ ]:
# Initialize RSI Mean-Reversion Results Collection System

# Create empty list to store all backtest results
grid_search_results = []

print("RSI Mean-Reversion Results Collection System Initialized")
print(f"   - Will test {len(rsi_combinations)} RSI parameter combinations")
print("   - Results will be stored in 'grid_search_results' list")

# Define what metrics we will collect (All TradingView-style metrics)
metrics_to_collect = [
    # Strategy Parameters
    "rsi_len",
    "oversold", 
    "overbought",
    
    # Return Metrics
    "total_return",
    "annualized_return",
    
    # Risk-Adjusted Return Metrics
    "sharpe_ratio",
    "sortino_ratio",
    "calmar_ratio",
    "omega_ratio",
    "information_ratio",
    "tail_ratio",
    "deflated_sharpe_ratio",
    
    # Risk Metrics
    "max_drawdown",
    "volatility",
    "ulcer_index",
    
    # Trade Performance Metrics
    "win_rate",
    "total_trades",
    "avg_trade_duration",
    "expectancy",
    "profit_factor", 
    "sqn",
    
    # Win/Loss Analysis
    "payoff_ratio",
    "largest_win",
    "largest_loss",
    "avg_win_amount",
    "avg_loss_amount",
    "winning_streak",
    "losing_streak",
    
    # Additional Ratios
    "recovery_factor",
    "gain_to_pain_ratio",
    "serenity_index"
]

print("Metrics to collect for each RSI combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")

print("Ready to start the RSI Mean-Reversion grid search!")


In [ ]:
# RSI MEAN-REVERSION GRID SEARCH ON TRAINING DATA

print("INITIATING RSI MEAN-REVERSION GRID SEARCH OPTIMIZATION")
print("=" * 70)
print(f"Testing Strategy: RSI Mean-Reversion (Oversold Entry / Overbought Exit)")
print(f"Training Period: {train_close.index[0].date()} \u2192 {train_close.index[-1].date()}")
print(f"Initial Capital: $100,000")
print(f"Transaction Costs: 0.05% per trade (fees + slippage)")
print(f"Optimization Metric: Sharpe Ratio (risk-adjusted returns)")
print("=" * 70)

total_combinations = len(rsi_combinations)
print(f"Total combinations to test: {total_combinations}")
print(f"Processing sequentially with progress every 1000 combos\n")

grid_search_results = []
successful_tests = 0
failed_tests = 0
skipped_low_trades = 0

train_close_vals = train_close.values.astype(float)
train_idx = train_close.index
years = max((train_idx[-1] - train_idx[0]).days / 365.25, 1e-9)

for combo_num, (rsi_len, oversold, overbought) in enumerate(rsi_combinations, 1):
    try:
        # Compute RSI
        rsi_vals = talib.RSI(train_close_vals, timeperiod=rsi_len)
        rsi_s = pd.Series(rsi_vals, index=train_idx)
        
        # Signal generation — RSI crosses UP through threshold
        entries_raw = (rsi_s.shift(1) <= oversold) & (rsi_s > oversold)
        exits_raw   = (rsi_s.shift(1) <= overbought) & (rsi_s > overbought)
        
        # Shift both by 1 bar for execution delay
        entries_shifted = entries_raw.shift(1)
        entries = pd.Series(np.where(entries_shifted.isna(), False, entries_shifted), index=train_idx, dtype=bool)
        
        exits_shifted = exits_raw.shift(1)
        exits = pd.Series(np.where(exits_shifted.isna(), False, exits_shifted), index=train_idx, dtype=bool)
        
        # Run backtest
        pf = vbt.Portfolio.from_signals(
            close=train_close,
            entries=entries,
            exits=exits,
            init_cash=100_000,
            fees=0.0005,
            slippage=0.0005,
            freq='D'
        )
        
        trades = pf.trades
        total_trades = len(trades)
        
        # Skip combos with < 10 trades
        if total_trades < 10:
            skipped_low_trades += 1
            continue
        
        trades_per_year = total_trades / years
        
        # Core metrics
        total_return = float(pf.total_return())
        annualized_return = float(pf.annualized_return(freq='D'))
        max_drawdown = float(pf.max_drawdown())
        volatility = float(pf.annualized_volatility(freq='D'))
        sharpe_ratio = float(pf.sharpe_ratio(freq='D'))
        sortino_ratio = float(pf.sortino_ratio(freq='D'))
        
        # Calmar ratio
        calmar_ratio = annualized_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan
        
        # Trade statistics
        win_rate_pct = np.nan
        profit_factor = np.nan
        expectancy = 0.0
        avg_win_amount = 0.0
        avg_loss_amount = 0.0
        largest_win = 0.0
        largest_loss = 0.0
        winning_streak = np.nan
        losing_streak = np.nan
        avg_trade_duration = np.nan
        sqn = np.nan
        
        tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
        if tr.size > 0:
            pos = tr[tr > 0]
            neg = tr[tr < 0]
            win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
            gains = pos.sum() if len(pos) else 0.0
            losses = abs(neg.sum()) if len(neg) else 0.0
            profit_factor = gains / losses if losses > 0 else np.inf
            expectancy = float(tr.mean())
            avg_win_amount = float(pos.mean()) if len(pos) else 0.0
            avg_loss_amount = float(abs(neg.mean())) if len(neg) else 0.0
            largest_win = float(pos.max()) if len(pos) else 0.0
            largest_loss = float(neg.min()) if len(neg) else 0.0
            sqn = float(tr.mean() / tr.std() * np.sqrt(len(tr))) if tr.std() > 0 else np.nan
            
            try:
                winning_streak = int(trades.winning_streak())
                losing_streak = int(trades.losing_streak())
            except:
                pass
        
        # Ulcer Index
        returns = pf.returns()
        cum = (1 + returns).cumprod()
        peak = cum.cummax()
        dd = (cum - peak) / peak
        ulcer_index = float(np.sqrt((dd.pow(2)).mean())) if len(dd) > 0 else np.nan
        
        payoff_ratio = (avg_win_amount / avg_loss_amount) if avg_loss_amount not in (0.0, np.nan) and avg_loss_amount > 0 else np.inf
        
        # Omega ratio (threshold = 0)
        rets = returns.values
        omega_ratio = float(rets[rets > 0].sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        
        # Recovery factor
        recovery_factor = total_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan
        
        # Gain to pain ratio
        gain_to_pain = float(rets.sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        
        # Serenity index = recovery_factor / ulcer_index
        serenity_index = recovery_factor / ulcer_index if ulcer_index and ulcer_index > 0 else np.nan
        
        grid_search_results.append({
            "rsi_len": rsi_len,
            "oversold": oversold,
            "overbought": overbought,
            "total_return": total_return,
            "annualized_return": annualized_return,
            "sharpe_ratio": sharpe_ratio,
            "sortino_ratio": sortino_ratio,
            "calmar_ratio": calmar_ratio,
            "omega_ratio": omega_ratio,
            "information_ratio": np.nan,
            "tail_ratio": np.nan,
            "deflated_sharpe_ratio": np.nan,
            "max_drawdown": max_drawdown,
            "volatility": volatility,
            "ulcer_index": ulcer_index,
            "win_rate": win_rate_pct,
            "total_trades": total_trades,
            "avg_trade_duration": avg_trade_duration,
            "expectancy": expectancy,
            "profit_factor": profit_factor,
            "sqn": sqn,
            "payoff_ratio": payoff_ratio,
            "largest_win": largest_win,
            "largest_loss": largest_loss,
            "avg_win_amount": avg_win_amount,
            "avg_loss_amount": avg_loss_amount,
            "winning_streak": winning_streak,
            "losing_streak": losing_streak,
            "recovery_factor": recovery_factor,
            "gain_to_pain_ratio": gain_to_pain,
            "serenity_index": serenity_index,
            "trades_per_year": trades_per_year
        })
        successful_tests += 1
        
    except Exception as e:
        failed_tests += 1
    
    # Progress every 1000 combos
    if combo_num % 1000 == 0:
        progress_pct = (combo_num / total_combinations) * 100
        print(f"\U0001f504 Progress: {combo_num}/{total_combinations} ({progress_pct:.1f}%) | \u2714 {successful_tests} successful | Skipped (low trades): {skipped_low_trades}")

# SUMMARY
print("\n" + "=" * 70)
print("RSI MEAN-REVERSION GRID SEARCH COMPLETED!")
print("=" * 70)
print(f"Total combinations attempted: {total_combinations}")
print(f"Successfully completed: {successful_tests}")
print(f"Skipped (< 10 trades): {skipped_low_trades}")
print(f"Failed: {failed_tests}")
print(f"Success rate: {(successful_tests/total_combinations)*100:.1f}%")
print(f"\n\u2714 Results stored in 'grid_search_results' ({len(grid_search_results)} entries)")

if successful_tests > 0:
    results_df_preview = pd.DataFrame(grid_search_results)
    
    # Display top 10 combinations
    print("\n" + "=" * 70)
    print("\U0001f3c6 TOP 10 COMBINATIONS (by In-Sample Sharpe Ratio)")
    print("=" * 70)
    
    top_10 = results_df_preview.nlargest(10, 'sharpe_ratio')
    for rank, (idx, row) in enumerate(top_10.iterrows(), 1):
        print(f"\n#{rank} - RSI(len={int(row['rsi_len'])}, OS={int(row['oversold'])}, OB={int(row['overbought'])})")
        print(f"   Sharpe Ratio:      {row['sharpe_ratio']:.3f}")
        print(f"   Total Return:      {row['total_return']:.2%}")
        print(f"   Annualized Return: {row['annualized_return']:.2%}")
        print(f"   Max Drawdown:      {row['max_drawdown']:.2%}")
        print(f"   Win Rate:          {row['win_rate']:.1f}%")
        print(f"   Profit Factor:     {row['profit_factor']:.2f}")
        print(f"   Total Trades:      {int(row['total_trades'])} ({row['trades_per_year']:.1f}/year)")
    
    print("\n" + "=" * 70)



In [ ]:
# TOP 5 OUT-OF-SAMPLE VALIDATION & COMPARISON TABLE

if 'FREQ' not in globals():
    FREQ = "1D"

results_df = pd.DataFrame(grid_search_results)

if results_df.empty:
    print("No results to validate.")
else:
    print("=" * 90)
    print("\U0001f3c6 TOP 5 STRATEGIES - OUT-OF-SAMPLE VALIDATION")
    print("=" * 90)
    print(f"Training Period: {train_close.index[0].date()} \u2192 {train_close.index[-1].date()}")
    print(f"Validation Period: {val_close.index[0].date()} \u2192 {val_close.index[-1].date()}")
    print("=" * 90)
    
    # Get top 5 by in-sample Sharpe
    top_5_strategies = results_df.nlargest(5, 'sharpe_ratio')
    
    # Results storage
    oos_results = []
    
    val_close_vals = val_close.values.astype(float)
    val_idx = val_close.index
    
    for idx_row, strategy in top_5_strategies.iterrows():
        rsi_len = int(strategy['rsi_len'])
        oversold = int(strategy['oversold'])
        overbought = int(strategy['overbought'])
        
        # Compute RSI on validation data
        rsi_vals = talib.RSI(val_close_vals, timeperiod=rsi_len)
        rsi_s = pd.Series(rsi_vals, index=val_idx)
        
        # Signal generation
        entries_raw = (rsi_s.shift(1) <= oversold) & (rsi_s > oversold)
        exits_raw   = (rsi_s.shift(1) <= overbought) & (rsi_s > overbought)
        
        # Shift by 1 bar
        entries_shifted = entries_raw.shift(1)
        entries = pd.Series(np.where(entries_shifted.isna(), False, entries_shifted), index=val_idx, dtype=bool)
        exits_shifted = exits_raw.shift(1)
        exits = pd.Series(np.where(exits_shifted.isna(), False, exits_shifted), index=val_idx, dtype=bool)
        
        # Run OOS backtest
        pf_val = vbt.Portfolio.from_signals(
            close=val_close,
            entries=entries,
            exits=exits,
            init_cash=100_000,
            fees=0.0005,
            slippage=0.0005,
            freq=FREQ
        )
        
        # OOS metrics
        oos_total_return = float(pf_val.total_return())
        oos_annualized_return = float(pf_val.annualized_return(freq=FREQ))
        oos_sharpe = float(pf_val.sharpe_ratio(freq=FREQ))
        oos_sortino = float(pf_val.sortino_ratio(freq=FREQ))
        oos_max_drawdown = float(pf_val.max_drawdown())
        oos_volatility = float(pf_val.annualized_volatility(freq=FREQ))
        
        trades = pf_val.trades
        oos_total_trades = len(trades)
        years = max((val_close.index[-1] - val_close.index[0]).days / 365.25, 1e-9)
        oos_trades_per_year = oos_total_trades / years
        
        oos_win_rate_pct = np.nan
        oos_profit_factor = np.nan
        oos_expectancy = 0.0
        
        if oos_total_trades > 0:
            tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
            if tr.size > 0:
                pos = tr[tr > 0]
                neg = tr[tr < 0]
                oos_win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
                gains = pos.sum() if len(pos) else 0.0
                losses = abs(neg.sum()) if len(neg) else 0.0
                oos_profit_factor = gains / losses if losses > 0 else np.inf
                oos_expectancy = float(tr.mean())
        
        # Store results
        oos_results.append({
            'Rank': len(oos_results) + 1,
            'RSI_Len': rsi_len,
            'Oversold': oversold,
            'Overbought': overbought,
            'IS_Sharpe': float(strategy['sharpe_ratio']),
            'IS_Return': float(strategy['total_return']),
            'IS_MaxDD': float(strategy['max_drawdown']),
            'IS_WinRate': float(strategy['win_rate']),
            'OOS_Sharpe': oos_sharpe,
            'OOS_Return': oos_total_return,
            'OOS_MaxDD': oos_max_drawdown,
            'OOS_WinRate': oos_win_rate_pct,
            'OOS_Trades': oos_total_trades,
            'OOS_ProfitFactor': oos_profit_factor,
            'Sharpe_Degradation': ((oos_sharpe - strategy['sharpe_ratio']) / abs(strategy['sharpe_ratio']) * 100) if strategy['sharpe_ratio'] != 0 else np.nan,
            'Return_Degradation': ((oos_total_return - strategy['total_return']) / abs(strategy['total_return']) * 100) if strategy['total_return'] != 0 else np.nan
        })
    
    # Create DataFrame
    oos_df = pd.DataFrame(oos_results)
    
    # Sort by OOS Sharpe (best to worst)
    oos_df_sorted = oos_df.sort_values('OOS_Sharpe', ascending=False).reset_index(drop=True)
    oos_df_sorted['OOS_Rank'] = range(1, len(oos_df_sorted) + 1)
    
    # Display comprehensive table
    print("\n\U0001f4ca COMPREHENSIVE COMPARISON TABLE (Sorted by OOS Sharpe)")
    print("=" * 90)
    
    display_df = pd.DataFrame({
        'IS\u2192OOS Rank': oos_df_sorted['Rank'].astype(str) + '\u2192' + oos_df_sorted['OOS_Rank'].astype(str),
        'Strategy': oos_df_sorted.apply(lambda x: f"RSI({int(x['RSI_Len'])},{int(x['Oversold'])},{int(x['Overbought'])})", axis=1),
        'IS Sharpe': oos_df_sorted['IS_Sharpe'].map('{:.3f}'.format),
        'OOS Sharpe': oos_df_sorted['OOS_Sharpe'].map('{:.3f}'.format),
        'Sharpe \u0394%': oos_df_sorted['Sharpe_Degradation'].map('{:+.1f}%'.format),
        'IS Return': oos_df_sorted['IS_Return'].map('{:.1%}'.format),
        'OOS Return': oos_df_sorted['OOS_Return'].map('{:.1%}'.format),
        'Return \u0394%': oos_df_sorted['Return_Degradation'].map('{:+.1f}%'.format),
        'OOS Trades': oos_df_sorted['OOS_Trades'].astype(int),
        'OOS WinRate': oos_df_sorted['OOS_WinRate'].map('{:.1f}%'.format),
        'OOS PF': oos_df_sorted['OOS_ProfitFactor'].map('{:.2f}'.format)
    })
    
    print(display_df.to_string(index=False))
    
    # Highlight best OOS performer
    best_oos = oos_df_sorted.iloc[0]
    print("\n" + "=" * 90)
    print(f"\u2728 BEST OUT-OF-SAMPLE PERFORMER")
    print("=" * 90)
    print(f"Strategy: RSI(len={int(best_oos['RSI_Len'])}, OS={int(best_oos['Oversold'])}, OB={int(best_oos['Overbought'])})")
    print(f"In-Sample Rank:        #{int(best_oos['Rank'])}")
    print(f"Out-of-Sample Rank:    #1")
    print(f"OOS Sharpe Ratio:      {best_oos['OOS_Sharpe']:.3f}")
    print(f"OOS Return:            {best_oos['OOS_Return']:.2%}")
    print(f"OOS Max Drawdown:      {best_oos['OOS_MaxDD']:.2%}")
    print(f"OOS Win Rate:          {best_oos['OOS_WinRate']:.1f}%")
    print(f"OOS Profit Factor:     {best_oos['OOS_ProfitFactor']:.2f}")
    print(f"OOS Total Trades:      {int(best_oos['OOS_Trades'])}")
    print(f"Sharpe Degradation:    {best_oos['Sharpe_Degradation']:+.1f}%")
    print("=" * 90)
    
    # ── Equity Curve Plots ──
    # Use the best IS strategy for equity curve visualization
    best_is = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    b_rsi_len = int(best_is['rsi_len'])
    b_oversold = int(best_is['oversold'])
    b_overbought = int(best_is['overbought'])
    
    # IS equity curve
    rsi_train = pd.Series(talib.RSI(train_close.values.astype(float), timeperiod=b_rsi_len), index=train_close.index)
    e_raw_tr = (rsi_train.shift(1) <= b_oversold) & (rsi_train > b_oversold)
    x_raw_tr = (rsi_train.shift(1) <= b_overbought) & (rsi_train > b_overbought)
    e_tr = pd.Series(np.where(e_raw_tr.shift(1).isna(), False, e_raw_tr.shift(1)), index=train_close.index, dtype=bool)
    x_tr = pd.Series(np.where(x_raw_tr.shift(1).isna(), False, x_raw_tr.shift(1)), index=train_close.index, dtype=bool)
    pf_train = vbt.Portfolio.from_signals(close=train_close, entries=e_tr, exits=x_tr,
                                           init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    
    # OOS equity curve
    rsi_val = pd.Series(talib.RSI(val_close.values.astype(float), timeperiod=b_rsi_len), index=val_close.index)
    e_raw_vl = (rsi_val.shift(1) <= b_oversold) & (rsi_val > b_oversold)
    x_raw_vl = (rsi_val.shift(1) <= b_overbought) & (rsi_val > b_overbought)
    e_vl = pd.Series(np.where(e_raw_vl.shift(1).isna(), False, e_raw_vl.shift(1)), index=val_close.index, dtype=bool)
    x_vl = pd.Series(np.where(x_raw_vl.shift(1).isna(), False, x_raw_vl.shift(1)), index=val_close.index, dtype=bool)
    pf_val2 = vbt.Portfolio.from_signals(close=val_close, entries=e_vl, exits=x_vl,
                                          init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    
    eq_train = (1 + pf_train.returns()).cumprod()
    eq_val = (1 + pf_val2.returns()).cumprod()
    
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(train_close.index, eq_train.values, label='Train (In-Sample)', color='blue', linewidth=1.5, alpha=0.8)
    ax.plot(val_close.index, eq_val.values, label='Validation (Out-of-Sample)', color='orange', linewidth=1.5, alpha=0.8)
    ax.axvline(x=train_close.index[-1], color='red', linestyle='--', alpha=0.5, label='Train/Val Split')
    ax.set_title(f'RSI(len={b_rsi_len}, OS={b_oversold}, OB={b_overbought}) - Equity Curves', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Cumulative Returns (normalized to 1)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()



PARAMETER SENSITIVITY ANALYSIS
------------------------------

This section evaluates how sensitive the strategy's performance is to small changes in each parameter.

For each parameter (`rsi_len`, `oversold`, `overbought`), we vary it ±15 around the best value while holding the other two fixed.

**Color coding**: Dark green (>+10% vs base), Light green (0-10%), Orange (-10-0%), Red (<-10%).

---

In [ ]:
# COMPACT PARAMETER SENSITIVITY ANALYSIS - BAR CHARTS

if results_df.empty:
    print("No results available for sensitivity analysis.")
else:
    # Use BEST IS strategy
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    rsi_len_base = int(best['rsi_len'])
    oversold_base = int(best['oversold'])
    overbought_base = int(best['overbought'])
    base_is_sharpe = float(best['sharpe_ratio'])
    base_oos_sharpe = np.nan
    
    # Try to get OOS sharpe from validation results
    try:
        rsi_val_check = pd.Series(talib.RSI(val_close.values.astype(float), timeperiod=rsi_len_base), index=val_close.index)
        e_raw_check = (rsi_val_check.shift(1) <= oversold_base) & (rsi_val_check > oversold_base)
        x_raw_check = (rsi_val_check.shift(1) <= overbought_base) & (rsi_val_check > overbought_base)
        e_check = pd.Series(np.where(e_raw_check.shift(1).isna(), False, e_raw_check.shift(1)), index=val_close.index, dtype=bool)
        x_check = pd.Series(np.where(x_raw_check.shift(1).isna(), False, x_raw_check.shift(1)), index=val_close.index, dtype=bool)
        pf_check = vbt.Portfolio.from_signals(close=val_close, entries=e_check, exits=x_check,
                                              init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        base_oos_sharpe = float(pf_check.sharpe_ratio(freq='D'))
    except:
        pass
    
    print(f"\U0001f52c PARAMETER SENSITIVITY ANALYSIS - Base: RSI(len={rsi_len_base}, OS={oversold_base}, OB={overbought_base})")
    print(f"IS Sharpe: {base_is_sharpe:.3f}" + (f" | OOS Sharpe: {base_oos_sharpe:.3f}" if not np.isnan(base_oos_sharpe) else ""))

    # Create sensitivity ranges (±15 around each parameter)
    rsi_len_sens = sorted(list(set(range(max(2, rsi_len_base - 15), rsi_len_base + 16))))
    oversold_sens = sorted(list(set(range(max(1, oversold_base - 15), oversold_base + 16))))
    overbought_sens = sorted(list(set(range(max(oversold_base + 11, overbought_base - 15), overbought_base + 16))))
    
    combos_rsi_len = [(rl, oversold_base, overbought_base) for rl in rsi_len_sens]
    combos_oversold = [(rsi_len_base, os, overbought_base) for os in oversold_sens]
    combos_overbought = [(rsi_len_base, oversold_base, ob) for ob in overbought_sens if ob > oversold_base + 10]
    all_combos = combos_rsi_len + combos_oversold + combos_overbought

    def eval_combo_both(rl: int, os: int, ob: int) -> dict:
        """Evaluate on both in-sample and out-of-sample"""
        # IN-SAMPLE
        rsi_is = pd.Series(talib.RSI(train_close.values.astype(float), timeperiod=rl), index=train_close.index)
        e_raw_is = (rsi_is.shift(1) <= os) & (rsi_is > os)
        x_raw_is = (rsi_is.shift(1) <= ob) & (rsi_is > ob)
        e_is = pd.Series(np.where(e_raw_is.shift(1).isna(), False, e_raw_is.shift(1)), index=train_close.index, dtype=bool)
        x_is = pd.Series(np.where(x_raw_is.shift(1).isna(), False, x_raw_is.shift(1)), index=train_close.index, dtype=bool)
        pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                           init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        
        # OUT-OF-SAMPLE
        rsi_oos = pd.Series(talib.RSI(val_close.values.astype(float), timeperiod=rl), index=val_close.index)
        e_raw_oos = (rsi_oos.shift(1) <= os) & (rsi_oos > os)
        x_raw_oos = (rsi_oos.shift(1) <= ob) & (rsi_oos > ob)
        e_oos = pd.Series(np.where(e_raw_oos.shift(1).isna(), False, e_raw_oos.shift(1)), index=val_close.index, dtype=bool)
        x_oos = pd.Series(np.where(x_raw_oos.shift(1).isna(), False, x_raw_oos.shift(1)), index=val_close.index, dtype=bool)
        pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                            init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        
        return {
            'rsi_len': rl, 'oversold': os, 'overbought': ob,
            'is_sharpe': float(pf_is.sharpe_ratio(freq='D')),
            'is_return': float(pf_is.total_return()),
            'is_maxdd': float(pf_is.max_drawdown()),
            'oos_sharpe': float(pf_oos.sharpe_ratio(freq='D')),
            'oos_return': float(pf_oos.total_return()),
            'oos_maxdd': float(pf_oos.max_drawdown()),
            'oos_trades': len(pf_oos.trades)
        }

    print(f"\n\U0001f504 Testing {len(all_combos)} parameter variations...")
    
    rows = []
    for combo in all_combos:
        try:
            rows.append(eval_combo_both(*combo))
        except Exception:
            pass

    if not rows:
        print("\u274c No sensitivity results computed.")
    else:
        sens_df = pd.DataFrame(rows)
        
        # Split by parameter variation type
        rsi_len_variations = sens_df[(sens_df['oversold'] == oversold_base) & (sens_df['overbought'] == overbought_base)].copy().sort_values('rsi_len')
        oversold_variations = sens_df[(sens_df['rsi_len'] == rsi_len_base) & (sens_df['overbought'] == overbought_base)].copy().sort_values('oversold')
        overbought_variations = sens_df[(sens_df['rsi_len'] == rsi_len_base) & (sens_df['oversold'] == oversold_base)].copy().sort_values('overbought')
        
        # Calculate degradations
        rsi_len_variations['is_sharpe_delta'] = ((rsi_len_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        oversold_variations['is_sharpe_delta'] = ((oversold_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        overbought_variations['is_sharpe_delta'] = ((overbought_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        
        if not np.isnan(base_oos_sharpe):
            rsi_len_variations['oos_sharpe_delta'] = ((rsi_len_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
            oversold_variations['oos_sharpe_delta'] = ((oversold_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
            overbought_variations['oos_sharpe_delta'] = ((overbought_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
        
        # CREATE BAR CHARTS - 2x3 grid
        fig, axes = plt.subplots(2, 3, figsize=(20, 10))
        fig.suptitle(f'Parameter Sensitivity Analysis - Base: RSI(len={rsi_len_base}, OS={oversold_base}, OB={overbought_base})', 
                     fontsize=16, fontweight='bold')
        
        # IN-SAMPLE BAR CHARTS
        # RSI Length IS
        colors1_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                      for x in rsi_len_variations['is_sharpe_delta']]
        axes[0, 0].bar(rsi_len_variations['rsi_len'], rsi_len_variations['is_sharpe_delta'], color=colors1_is, edgecolor='black', linewidth=0.5)
        axes[0, 0].axhline(0, color='black', linewidth=1.5)
        axes[0, 0].axvline(rsi_len_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={rsi_len_base}')
        axes[0, 0].set_xlabel('RSI Length', fontsize=11, fontweight='bold')
        axes[0, 0].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
        axes[0, 0].set_title('RSI Length - In-Sample', fontsize=12, fontweight='bold')
        axes[0, 0].grid(axis='y', alpha=0.3)
        axes[0, 0].legend(fontsize=10)
        
        # Oversold IS
        colors2_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                      for x in oversold_variations['is_sharpe_delta']]
        axes[0, 1].bar(oversold_variations['oversold'], oversold_variations['is_sharpe_delta'], color=colors2_is, edgecolor='black', linewidth=0.5)
        axes[0, 1].axhline(0, color='black', linewidth=1.5)
        axes[0, 1].axvline(oversold_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={oversold_base}')
        axes[0, 1].set_xlabel('Oversold Threshold', fontsize=11, fontweight='bold')
        axes[0, 1].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
        axes[0, 1].set_title('Oversold - In-Sample', fontsize=12, fontweight='bold')
        axes[0, 1].grid(axis='y', alpha=0.3)
        axes[0, 1].legend(fontsize=10)
        
        # Overbought IS
        colors3_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                      for x in overbought_variations['is_sharpe_delta']]
        axes[0, 2].bar(overbought_variations['overbought'], overbought_variations['is_sharpe_delta'], color=colors3_is, edgecolor='black', linewidth=0.5)
        axes[0, 2].axhline(0, color='black', linewidth=1.5)
        axes[0, 2].axvline(overbought_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={overbought_base}')
        axes[0, 2].set_xlabel('Overbought Threshold', fontsize=11, fontweight='bold')
        axes[0, 2].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
        axes[0, 2].set_title('Overbought - In-Sample', fontsize=12, fontweight='bold')
        axes[0, 2].grid(axis='y', alpha=0.3)
        axes[0, 2].legend(fontsize=10)
        
        # OUT-OF-SAMPLE BAR CHARTS
        if not np.isnan(base_oos_sharpe):
            # RSI Length OOS
            colors1_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                          for x in rsi_len_variations['oos_sharpe_delta']]
            axes[1, 0].bar(rsi_len_variations['rsi_len'], rsi_len_variations['oos_sharpe_delta'], color=colors1_oos, edgecolor='black', linewidth=0.5)
            axes[1, 0].axhline(0, color='black', linewidth=1.5)
            axes[1, 0].axvline(rsi_len_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={rsi_len_base}')
            axes[1, 0].set_xlabel('RSI Length', fontsize=11, fontweight='bold')
            axes[1, 0].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[1, 0].set_title('RSI Length - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, 0].grid(axis='y', alpha=0.3)
            axes[1, 0].legend(fontsize=10)
            
            # Oversold OOS
            colors2_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                          for x in oversold_variations['oos_sharpe_delta']]
            axes[1, 1].bar(oversold_variations['oversold'], oversold_variations['oos_sharpe_delta'], color=colors2_oos, edgecolor='black', linewidth=0.5)
            axes[1, 1].axhline(0, color='black', linewidth=1.5)
            axes[1, 1].axvline(oversold_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={oversold_base}')
            axes[1, 1].set_xlabel('Oversold Threshold', fontsize=11, fontweight='bold')
            axes[1, 1].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[1, 1].set_title('Oversold - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, 1].grid(axis='y', alpha=0.3)
            axes[1, 1].legend(fontsize=10)
            
            # Overbought OOS
            colors3_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                          for x in overbought_variations['oos_sharpe_delta']]
            axes[1, 2].bar(overbought_variations['overbought'], overbought_variations['oos_sharpe_delta'], color=colors3_oos, edgecolor='black', linewidth=0.5)
            axes[1, 2].axhline(0, color='black', linewidth=1.5)
            axes[1, 2].axvline(overbought_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={overbought_base}')
            axes[1, 2].set_xlabel('Overbought Threshold', fontsize=11, fontweight='bold')
            axes[1, 2].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[1, 2].set_title('Overbought - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, 2].grid(axis='y', alpha=0.3)
            axes[1, 2].legend(fontsize=10)
        
        plt.tight_layout()
        plt.show()
        
        # COMPACT SUMMARY TABLE
        print("\n\U0001f4cb SENSITIVITY SUMMARY")
        print("=" * 80)
        
        summary_data = []
        for param_name, variations, param_col in [('RSI_Len', rsi_len_variations, 'rsi_len'), 
                                                   ('Oversold', oversold_variations, 'oversold'), 
                                                   ('Overbought', overbought_variations, 'overbought')]:
            summary_data.append({
                'Parameter': param_name,
                'IS Range': f"{variations['is_sharpe'].min():.3f} - {variations['is_sharpe'].max():.3f}",
                'IS Max \u0394%': f"{variations['is_sharpe_delta'].min():.1f}%",
                'OOS Range': f"{variations['oos_sharpe'].min():.3f} - {variations['oos_sharpe'].max():.3f}" if not np.isnan(base_oos_sharpe) else 'N/A',
                'OOS Max \u0394%': f"{variations['oos_sharpe_delta'].min():.1f}%" if not np.isnan(base_oos_sharpe) and 'oos_sharpe_delta' in variations else 'N/A',
                'Sensitivity': '\u26a0\ufe0f HIGH' if abs(variations['is_sharpe_delta'].min()) > 40 else '\u2705 LOW'
            })
        
        summary_df = pd.DataFrame(summary_data)
        print(summary_df.to_string(index=False))
        
        print("\n\u2705 Analysis Complete! Green = Robust, Red = Fragile")



In [ ]:
# ════════════════════════════════════════════════════════════════════
# UNIVERSAL STRATEGY EXPORT — Data Files + PDF Tearsheet
# ════════════════════════════════════════════════════════════════════
# INSTRUCTIONS:
#   1. Paste at the END of any strategy notebook
#   2. Edit STRATEGY_NAME and PARAM_COLS below
#   3. Run — exports structured data + a comprehensive PDF report
# ════════════════════════════════════════════════════════════════════

import os, sys, json, datetime, hashlib, platform
from matplotlib.backends.backend_pdf import PdfPages

# ═══ EDIT THESE LINES ═══════════════════════════════════════
STRATEGY_NAME = "RSI_MeanReversion"                                    # ← EDIT
PARAM_COLS    = ["rsi_len", "oversold", "overbought"]     # ← EDIT
NOTES         = ""                                                   # ← Optional run notes
# ════════════════════════════════════════════════════════════════

INIT_CASH = 100_000
FEES      = 0.0005
SLIPPAGE  = 0.0005
FREQ      = 'D'

# ── Google Drive mount ──
EXPORT_DIR = "/content/strategy_exports"
IN_COLAB = 'google.colab' in sys.modules
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    EXPORT_DIR = "/content/drive/MyDrive/strategy_exports"
    IN_COLAB = True
    print("\u2705 Google Drive mounted")
except:
    print("\u26a0\ufe0f Local mode — exports to ./strategy_exports")

RUN_TIMESTAMP = datetime.datetime.now()
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%d_%H%M%S")

# ── Folder structure ──
STRAT_DIR   = os.path.join(EXPORT_DIR, STRATEGY_NAME, TICKER)
LATEST_DIR  = os.path.join(STRAT_DIR, "latest")
ARCHIVE_DIR = os.path.join(STRAT_DIR, "archive")
os.makedirs(LATEST_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

# ════════════════════════════════════════════════════════════════
# Signal function — auto-detects strategy type
# ════════════════════════════════════════════════════════════════
def _compute_signals(price_s, params, high_s=None, low_s=None):
    idx = price_s.index; vals = price_s.values.astype(float)
    if STRATEGY_NAME.startswith("MACD"):
        ml, sl, _ = talib.MACD(vals, fastperiod=params['fast_period'], slowperiod=params['slow_period'], signalperiod=params['signal_period'])
        ms, ss = pd.Series(ml, index=idx), pd.Series(sl, index=idx)
        e_raw = (ms.shift(1) <= ss.shift(1)) & (ms > ss)
        x_raw = (ms.shift(1) >= ss.shift(1)) & (ms < ss)
    elif STRATEGY_NAME.startswith("RSI"):
        rsi_s = pd.Series(talib.RSI(vals, timeperiod=params['rsi_len']), index=idx)
        e_raw = (rsi_s.shift(1) <= params['oversold']) & (rsi_s > params['oversold'])
        x_raw = (rsi_s.shift(1) <= params['overbought']) & (rsi_s > params['overbought'])
    elif STRATEGY_NAME.startswith("EMA"):
        fv = pd.Series(talib.EMA(vals, timeperiod=params['fast_ema']), index=idx)
        sv = pd.Series(talib.EMA(vals, timeperiod=params['slow_ema']), index=idx)
        tv = pd.Series(talib.SMA(vals, timeperiod=params['trend_filter']), index=idx)
        cs = pd.Series(vals, index=idx)
        e_raw = ((fv.shift(1) <= sv.shift(1)) & (fv > sv) & (cs > tv))
        x_raw = ((fv.shift(1) >= sv.shift(1)) & (fv < sv))
    elif STRATEGY_NAME.startswith("Triple") or STRATEGY_NAME.startswith("TEMA"):
        e1 = pd.Series(vbt.MA.run(price_s, params['ema1_period'], ewm=True).ma.values.flatten(), index=idx)
        e2 = pd.Series(vbt.MA.run(price_s, params['ema2_period'], ewm=True).ma.values.flatten(), index=idx)
        e3 = pd.Series(vbt.MA.run(price_s, params['ema3_period'], ewm=True).ma.values.flatten(), index=idx)
        e_raw = (e1.vbt.crossed_above(e2) | e1.vbt.crossed_above(e3) | e2.vbt.crossed_above(e3))
        x_raw = (e1.vbt.crossed_below(e2) | e1.vbt.crossed_below(e3) | e2.vbt.crossed_below(e3))
    elif STRATEGY_NAME.startswith("Donchian") or STRATEGY_NAME.startswith("DC"):
        h_v = high_s.values.astype(float) if high_s is not None else vals
        l_v = low_s.values.astype(float) if low_s is not None else vals
        uc = pd.Series(talib.MAX(h_v, timeperiod=params['entry_period']), index=idx).shift(1)
        lc = pd.Series(talib.MIN(l_v, timeperiod=params['exit_period']), index=idx).shift(1)
        tf = pd.Series(talib.SMA(vals, timeperiod=params['filter_period']), index=idx).shift(1)
        cs = pd.Series(vals, index=idx)
        e_raw = (cs > uc) & (cs > tf); x_raw = (cs < lc)
    else:
        raise ValueError(f"Unknown strategy: {STRATEGY_NAME}")
    entries = pd.Series(np.where(e_raw.shift(1).isna(), False, e_raw.shift(1)), index=idx, dtype=bool)
    exits = pd.Series(np.where(x_raw.shift(1).isna(), False, x_raw.shift(1)), index=idx, dtype=bool)
    return entries, exits

# ════════════════════════════════════════════════════════════════
# Build portfolios
# ════════════════════════════════════════════════════════════════
results_df = pd.DataFrame(grid_search_results)
best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
best_params = {col: int(best[col]) for col in PARAM_COLS}
param_str = ", ".join([f"{k}={v}" for k, v in best_params.items()])

if isinstance(stock_data.columns, pd.MultiIndex):
    full_close = stock_data[('Close', TICKER)].astype(float).squeeze()
else:
    full_close = stock_data['Close'].astype(float).squeeze()
full_close.name = 'price'

split_idx = int(len(full_close) * 0.60)
train_close = full_close.iloc[:split_idx].copy()
val_close = full_close.iloc[split_idx:].copy()

# High/Low for Donchian
high_s, low_s = None, None
if STRATEGY_NAME.startswith("Donchian") or STRATEGY_NAME.startswith("DC"):
    if isinstance(stock_data.columns, pd.MultiIndex):
        high_s = stock_data[('High', TICKER)].astype(float).squeeze()
        low_s = stock_data[('Low', TICKER)].astype(float).squeeze()
    else:
        high_s = stock_data['High'].astype(float).squeeze()
        low_s = stock_data['Low'].astype(float).squeeze()

# Full sample
e_full, x_full = _compute_signals(full_close, best_params, high_s, low_s)
pf_full = vbt.Portfolio.from_signals(close=full_close, entries=e_full, exits=x_full,
                                      init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
# IS
e_is, x_is = _compute_signals(train_close, best_params,
                                high_s.iloc[:split_idx] if high_s is not None else None,
                                low_s.iloc[:split_idx] if low_s is not None else None)
pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
# OOS
e_oos, x_oos = _compute_signals(val_close, best_params,
                                  high_s.iloc[split_idx:] if high_s is not None else None,
                                  low_s.iloc[split_idx:] if low_s is not None else None)
pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                     init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)
# Buy & Hold
bh_e = pd.Series(False, index=full_close.index, dtype=bool); bh_e.iloc[0] = True
bh_x = pd.Series(False, index=full_close.index, dtype=bool)
pf_bh = vbt.Portfolio.from_signals(close=full_close, entries=bh_e, exits=bh_x,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=FREQ)

# ── Extract metrics ──
trades_obj = pf_full.trades
tr = np.asarray(trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns).ravel()
pnl = np.asarray(trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else trades_obj.pnl).ravel()
pos, neg = tr[tr > 0], tr[tr < 0]
years_full = max((full_close.index[-1] - full_close.index[0]).days / 365.25, 1e-9)
daily_rets = pf_full.returns()

def safe(fn, default=None):
    try: return float(fn())
    except: return default

M = {  # Master metrics dict
    'total_return': safe(pf_full.total_return), 'ann_return': safe(lambda: pf_full.annualized_return(freq=FREQ)),
    'sharpe': safe(lambda: pf_full.sharpe_ratio(freq=FREQ)), 'sortino': safe(lambda: pf_full.sortino_ratio(freq=FREQ)),
    'max_dd': safe(pf_full.max_drawdown), 'volatility': safe(lambda: pf_full.annualized_volatility(freq=FREQ)),
    'calmar': safe(lambda: pf_full.annualized_return(freq=FREQ)) / abs(safe(pf_full.max_drawdown)) if abs(safe(pf_full.max_drawdown, 0)) > 1e-9 else None,
    'trades': len(trades_obj), 'trades_yr': len(trades_obj) / years_full,
    'win_rate': float(len(pos) / len(tr) * 100) if len(tr) > 0 else None,
    'pf': float(pos.sum() / abs(neg.sum())) if len(neg) > 0 and abs(neg.sum()) > 0 else None,
    'expectancy': float(tr.mean()) if len(tr) > 0 else None,
    'avg_win': float(pos.mean()) if len(pos) > 0 else None, 'avg_loss': float(neg.mean()) if len(neg) > 0 else None,
    'largest_win': float(pos.max()) if len(pos) > 0 else None, 'largest_loss': float(neg.min()) if len(neg) > 0 else None,
    'payoff': float(abs(pos.mean() / neg.mean())) if len(pos) > 0 and len(neg) > 0 else None,
    'is_sharpe': safe(lambda: pf_is.sharpe_ratio(freq=FREQ)), 'is_return': safe(pf_is.total_return),
    'is_dd': safe(pf_is.max_drawdown), 'is_trades': len(pf_is.trades),
    'oos_sharpe': safe(lambda: pf_oos.sharpe_ratio(freq=FREQ)), 'oos_return': safe(pf_oos.total_return),
    'oos_dd': safe(pf_oos.max_drawdown), 'oos_trades': len(pf_oos.trades),
    'bh_return': safe(pf_bh.total_return), 'bh_sharpe': safe(lambda: pf_bh.sharpe_ratio(freq=FREQ)),
    'bh_dd': safe(pf_bh.max_drawdown),
}

# ════════════════════════════════════════════════════════════════
# 1. SAVE STRUCTURED DATA FILES
# ════════════════════════════════════════════════════════════════
export_json = {
    "metadata": {
        "run_id": RUN_ID, "export_timestamp": RUN_TIMESTAMP.isoformat(),
        "export_date_human": RUN_TIMESTAMP.strftime("%B %d, %Y at %I:%M %p"),
        "strategy_name": STRATEGY_NAME, "strategy_family": STRATEGY_NAME.split("_")[0],
        "ticker": TICKER,
        "instrument_type": ("crypto" if "-USD" in TICKER and TICKER.replace("-USD","").isalpha()
                           else "forex" if "/" in TICKER or (len(TICKER) == 6 and TICKER.isalpha())
                           else "equity/etf"),
        "data_source": "yfinance", "data_interval": "1d", "currency": "USD",
        "start_date": str(full_close.index[0].date()), "end_date": str(full_close.index[-1].date()),
        "total_bars": len(full_close), "total_years": round(years_full, 2),
        "train_start": str(train_close.index[0].date()), "train_end": str(train_close.index[-1].date()),
        "train_bars": len(train_close), "val_start": str(val_close.index[0].date()),
        "val_end": str(val_close.index[-1].date()), "val_bars": len(val_close), "train_ratio": 0.60,
        "init_cash": INIT_CASH, "fees_pct": FEES, "slippage_pct": SLIPPAGE, "frequency": FREQ,
        "first_close": round(float(full_close.iloc[0]), 4), "last_close": round(float(full_close.iloc[-1]), 4),
        "python_version": sys.version.split()[0], "environment": "colab_pro" if IN_COLAB else "local",
        "grid_combos_tested": len(results_df), "param_columns": PARAM_COLS, "notes": NOTES,
    },
    "best_params": best_params,
    "metrics_full_sample": {k: v for k, v in M.items() if not k.startswith('is_') and not k.startswith('oos_') and not k.startswith('bh_')},
    "metrics_in_sample": {k.replace('is_',''): v for k, v in M.items() if k.startswith('is_')},
    "metrics_out_of_sample": {k.replace('oos_',''): v for k, v in M.items() if k.startswith('oos_')},
    "metrics_buy_hold": {k.replace('bh_',''): v for k, v in M.items() if k.startswith('bh_')},
    "grid_search_summary": {
        "top5": results_df.nlargest(5, 'sharpe_ratio')[PARAM_COLS + ['sharpe_ratio','total_return','max_drawdown']].to_dict('records'),
    }
}

# Save JSON
with open(os.path.join(LATEST_DIR, "summary.json"), 'w') as f:
    json.dump(export_json, f, indent=2, default=str)
with open(os.path.join(ARCHIVE_DIR, f"{RUN_ID}_summary.json"), 'w') as f:
    json.dump(export_json, f, indent=2, default=str)
print(f"\u2705 summary.json")

# Save CSVs
pd.DataFrame({'date': full_close.index.strftime('%Y-%m-%d'), 'strategy_return': daily_rets.values,
              'close': full_close.values, 'portfolio_value': pf_full.value().values
}).to_csv(os.path.join(LATEST_DIR, "daily_returns.csv"), index=False)
print(f"\u2705 daily_returns.csv")

pd.DataFrame({'trade_num': range(1, len(tr)+1), 'return_pct': tr*100, 'pnl_usd': pnl,
              'cumulative_pnl': np.cumsum(pnl), 'is_winner': tr > 0
}).to_csv(os.path.join(LATEST_DIR, "trades.csv"), index=False)
print(f"\u2705 trades.csv")

results_df.to_csv(os.path.join(LATEST_DIR, "grid_results.csv"), index=False)
print(f"\u2705 grid_results.csv")

# Run log
log_path = os.path.join(EXPORT_DIR, "run_log.csv")
log_entry = pd.DataFrame([{
    "run_id": RUN_ID, "timestamp": RUN_TIMESTAMP.isoformat(), "strategy": STRATEGY_NAME,
    "ticker": TICKER, "best_params": str(best_params),
    "sharpe_full": round(M['sharpe'] or 0, 4), "sharpe_is": round(M['is_sharpe'] or 0, 4),
    "sharpe_oos": round(M['oos_sharpe'] or 0, 4), "total_return": round(M['total_return'] or 0, 4),
    "max_drawdown": round(M['max_dd'] or 0, 4), "total_trades": M['trades'],
    "win_rate": round(M['win_rate'] or 0, 1), "profit_factor": round(M['pf'] or 0, 2) if M['pf'] else None,
    "notes": NOTES, "export_path": STRAT_DIR,
}])
if os.path.exists(log_path):
    log_combined = pd.concat([pd.read_csv(log_path), log_entry], ignore_index=True)
else:
    log_combined = log_entry
log_combined.to_csv(log_path, index=False)
print(f"\u2705 run_log.csv ({len(log_combined)} runs)")

# ════════════════════════════════════════════════════════════════
# 2. GENERATE PDF TEARSHEET
# ════════════════════════════════════════════════════════════════
pdf_path = os.path.join(LATEST_DIR, "tearsheet.pdf")
pdf_archive = os.path.join(ARCHIVE_DIR, f"{RUN_ID}_tearsheet.pdf")

fmt = lambda v, d=2: f"{v:.{d}f}" if v is not None and not np.isnan(v) else "N/A"
fmtp = lambda v: f"{v:.2%}" if v is not None and not np.isnan(v) else "N/A"

with PdfPages(pdf_path) as pdf:

    # ── PAGE 1: Summary Metrics ──
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')
    fig.patch.set_facecolor('#0d0d1a')

    # Title
    ax.text(0.5, 0.95, f"STRATEGY TEARSHEET", ha='center', va='top', fontsize=22,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.90, f"{STRATEGY_NAME}  |  {TICKER}  |  {param_str}", ha='center', va='top',
            fontsize=13, color='#8888cc', transform=ax.transAxes, family='monospace')
    ax.text(0.5, 0.86, f"{full_close.index[0].date()} to {full_close.index[-1].date()}  |  {len(full_close)} bars  |  Run: {RUN_ID}",
            ha='center', va='top', fontsize=10, color='#6a6a8a', transform=ax.transAxes)

    # Metrics table
    rows = [
        ["METRIC", "FULL SAMPLE", "IN-SAMPLE", "OUT-OF-SAMPLE", "BUY & HOLD"],
        ["Total Return", fmtp(M['total_return']), fmtp(M['is_return']), fmtp(M['oos_return']), fmtp(M['bh_return'])],
        ["Sharpe Ratio", fmt(M['sharpe'], 3), fmt(M['is_sharpe'], 3), fmt(M['oos_sharpe'], 3), fmt(M['bh_sharpe'], 3)],
        ["Sortino Ratio", fmt(M['sortino'], 3), "—", "—", "—"],
        ["Max Drawdown", fmtp(M['max_dd']), fmtp(M['is_dd']), fmtp(M['oos_dd']), fmtp(M['bh_dd'])],
        ["Calmar Ratio", fmt(M['calmar'], 3), "—", "—", "—"],
        ["Volatility", fmtp(M['volatility']), "—", "—", "—"],
        ["Win Rate", f"{M['win_rate']:.1f}%" if M['win_rate'] else "N/A", "—", "—", "—"],
        ["Profit Factor", fmt(M['pf']), "—", "—", "—"],
        ["Expectancy", fmt(M['expectancy'], 4), "—", "—", "—"],
        ["Payoff Ratio", fmt(M['payoff']), "—", "—", "—"],
        ["Total Trades", str(M['trades']), str(M['is_trades']), str(M['oos_trades']), "1"],
        ["Trades/Year", fmt(M['trades_yr'], 1), "—", "—", "—"],
        ["Avg Win", fmtp(M['avg_win']), "—", "—", "—"],
        ["Avg Loss", fmtp(M['avg_loss']), "—", "—", "—"],
        ["Largest Win", fmtp(M['largest_win']), "—", "—", "—"],
        ["Largest Loss", fmtp(M['largest_loss']), "—", "—", "—"],
    ]

    table = ax.table(cellText=rows[1:], colLabels=rows[0], cellLoc='center', loc='center',
                     bbox=[0.02, 0.03, 0.96, 0.78])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor('#2a2a4a')
        if row == 0:
            cell.set_facecolor('#1a1a3e')
            cell.set_text_props(color='white', fontweight='bold', fontsize=8)
        else:
            cell.set_facecolor('#111128' if row % 2 == 0 else '#0d0d1a')
            cell.set_text_props(color='#d0d0e8', fontsize=8, family='monospace')
        cell.set_height(0.055)

    pdf.savefig(fig, facecolor='#0d0d1a')
    plt.close(fig)

    # ── PAGE 2: Equity Curve + Drawdown ──
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8.5), gridspec_kw={'height_ratios': [3, 1]})
    fig.patch.set_facecolor('#0d0d1a')
    fig.suptitle(f'{STRATEGY_NAME} on {TICKER} — Equity & Drawdown', fontsize=14, fontweight='bold', color='white')

    eq_strat = pf_full.value(); eq_bh = pf_bh.value()
    ax1.plot(full_close.index[:split_idx], eq_strat.iloc[:split_idx].values, color='#3498db', linewidth=1.5, label='Strategy (IS)')
    ax1.plot(full_close.index[split_idx:], eq_strat.iloc[split_idx:].values, color='#e67e22', linewidth=1.5, label='Strategy (OOS)')
    ax1.plot(full_close.index, eq_bh.values, color='gray', linewidth=1, alpha=0.5, linestyle='--', label='Buy & Hold')
    ax1.axvline(x=full_close.index[split_idx], color='red', linestyle=':', alpha=0.4)
    ax1.set_facecolor('#0d0d1a'); ax1.tick_params(colors='#8888aa'); ax1.grid(True, alpha=0.15, color='#333')
    ax1.set_ylabel('Portfolio Value ($)', color='#8888aa'); ax1.legend(fontsize=8, facecolor='#111128', edgecolor='#333', labelcolor='#d0d0e8')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    for spine in ax1.spines.values(): spine.set_color('#2a2a4a')

    # Stats box on equity chart
    stats_text = f"Sharpe: {fmt(M['sharpe'],3)}  |  Return: {fmtp(M['total_return'])}  |  MaxDD: {fmtp(M['max_dd'])}  |  WR: {M['win_rate']:.1f}%  |  PF: {fmt(M['pf'])}"
    ax1.text(0.5, 0.02, stats_text, transform=ax1.transAxes, fontsize=8, ha='center', color='#aaa', family='monospace',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#111128', edgecolor='#2a2a4a', alpha=0.9))

    dd = pf_full.drawdown() * 100
    ax2.fill_between(full_close.index, dd.values, 0, color='#e74c3c', alpha=0.5)
    ax2.set_facecolor('#0d0d1a'); ax2.tick_params(colors='#8888aa'); ax2.grid(True, alpha=0.15, color='#333')
    ax2.set_ylabel('Drawdown %', color='#8888aa'); ax2.set_xlabel('Date', color='#8888aa')
    for spine in ax2.spines.values(): spine.set_color('#2a2a4a')

    plt.tight_layout()
    pdf.savefig(fig, facecolor='#0d0d1a')
    plt.close(fig)

    # ── PAGE 3: Trade Analysis ──
    if len(tr) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
        fig.patch.set_facecolor('#0d0d1a')
        fig.suptitle(f'Trade-by-Trade Analysis — {len(tr)} Trades', fontsize=14, fontweight='bold', color='white')

        n = len(tr)
        colors_bar = ['#2ca02c' if r > 0 else '#e74c3c' for r in tr]
        colors_pnl = ['#2ca02c' if p > 0 else '#e74c3c' for p in pnl]

        for a in axes.flat:
            a.set_facecolor('#0d0d1a'); a.tick_params(colors='#8888aa'); a.grid(True, alpha=0.15, color='#333')
            for spine in a.spines.values(): spine.set_color('#2a2a4a')

        axes[0,0].bar(range(n), tr*100, color=colors_bar, edgecolor='none', width=0.8)
        axes[0,0].axhline(np.mean(tr)*100, color='#3498db', linestyle='--', linewidth=1.5)
        axes[0,0].set_title('Trade Returns (%)', color='white', fontsize=10); axes[0,0].set_xlabel('Trade #', color='#8888aa')

        axes[0,1].bar(range(n), pnl, color=colors_pnl, edgecolor='none', width=0.8)
        axes[0,1].axhline(np.mean(pnl), color='#3498db', linestyle='--', linewidth=1.5)
        axes[0,1].set_title('Trade P&L ($)', color='white', fontsize=10); axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

        cum_pnl = np.cumsum(pnl)
        axes[1,0].plot(range(1, n+1), cum_pnl, color='#3498db', linewidth=2, marker='o', markersize=3)
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl>=0, alpha=0.15, color='green')
        axes[1,0].fill_between(range(1, n+1), cum_pnl, 0, where=cum_pnl<0, alpha=0.15, color='red')
        axes[1,0].set_title('Cumulative P&L', color='white', fontsize=10); axes[1,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

        axes[1,1].hist(tr*100, bins=min(30, max(10, n//3)), color='#3498db', edgecolor='#1a1a2e', alpha=0.8)
        axes[1,1].axvline(np.mean(tr)*100, color='#e74c3c', linestyle='--', linewidth=2)
        axes[1,1].set_title('Return Distribution', color='white', fontsize=10); axes[1,1].set_xlabel('Return %', color='#8888aa')

        plt.tight_layout()
        pdf.savefig(fig, facecolor='#0d0d1a')
        plt.close(fig)

    # ── PAGE 4: Monte Carlo FTMO ──
    dr = daily_rets.values.ravel(); dr = dr[~np.isnan(dr)]
    N_SIM = 5000; DAYS = 30; ACCOUNT = 100000
    np.random.seed(42)
    n_passed = n_blown_t = n_blown_d = 0
    final_eqs = np.zeros(N_SIM); sample_paths = []

    for s in range(N_SIM):
        eq = ACCOUNT; path = [ACCOUNT]
        sim_rets = np.random.choice(dr, size=DAYS, replace=True)
        blown = False
        for d in range(DAYS):
            day_start = eq; eq *= (1 + sim_rets[d]); path.append(eq)
            if (eq - day_start)/ACCOUNT < -0.05: n_blown_d += 1; blown = True; break
            if (eq - ACCOUNT)/ACCOUNT < -0.10: n_blown_t += 1; blown = True; break
            if (eq - ACCOUNT)/ACCOUNT >= 0.10: n_passed += 1; blown = True; break
        while len(path) < DAYS + 1: path.append(path[-1])
        final_eqs[s] = path[-1]
        if s < 150: sample_paths.append(path)

    n_still = N_SIM - n_passed - n_blown_t - n_blown_d
    pass_rate = n_passed / N_SIM * 100

    fig, ax = plt.subplots(figsize=(11, 8.5))
    fig.patch.set_facecolor('#0d0d1a'); ax.set_facecolor('#0d0d1a')
    for spine in ax.spines.values(): spine.set_color('#2a2a4a')
    ax.tick_params(colors='#8888aa'); ax.grid(True, alpha=0.15, color='#333')

    for path in sample_paths:
        c = '#2ca02c' if path[-1] >= 110000 else ('#e74c3c' if path[-1] <= 90000 else '#555555')
        ax.plot(range(DAYS+1), path, color=c, alpha=0.25, linewidth=0.5)
    ax.axhline(110000, color='#2ca02c', linestyle='--', linewidth=2.5, label=f'10% Target ($110k)')
    ax.axhline(90000, color='#e74c3c', linestyle='--', linewidth=2.5, label=f'10% Max Loss ($90k)')
    ax.axhline(100000, color='white', linestyle='-', linewidth=0.5, alpha=0.3)

    verdict = "FAVORABLE" if pass_rate >= 50 else "POSSIBLE" if pass_rate >= 25 else "CHALLENGING" if pass_rate >= 10 else "UNLIKELY"
    emoji = "\U0001f7e2" if pass_rate >= 50 else "\U0001f7e1" if pass_rate >= 25 else "\U0001f7e0" if pass_rate >= 10 else "\U0001f534"

    ax.set_title(f'FTMO Monte Carlo — {N_SIM:,} Simulations  |  Pass Rate: {pass_rate:.1f}% ({verdict})', fontsize=14, fontweight='bold', color='white')
    ax.set_xlabel('Trading Day', color='#8888aa'); ax.set_ylabel('Equity ($)', color='#8888aa')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=10, facecolor='#111128', edgecolor='#333', labelcolor='#d0d0e8')

    mc_text = (f"Passed: {n_passed:,} ({n_passed/N_SIM*100:.1f}%)  |  "
               f"Blown Total: {n_blown_t:,}  |  Blown Daily: {n_blown_d:,}  |  "
               f"Still Trading: {n_still:,}  |  Median Final: ${np.median(final_eqs):,.0f}")
    ax.text(0.5, 0.02, mc_text, transform=ax.transAxes, fontsize=8, ha='center', color='#aaa', family='monospace',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#111128', edgecolor='#2a2a4a', alpha=0.9))

    plt.tight_layout()
    pdf.savefig(fig, facecolor='#0d0d1a')
    plt.close(fig)

# Copy PDF to archive
import shutil
shutil.copy2(pdf_path, pdf_archive)

# Add MC results to JSON
mc_data = {"pass_rate": round(pass_rate, 1), "n_simulations": N_SIM, "n_passed": n_passed,
           "n_blown_total": n_blown_t, "n_blown_daily": n_blown_d, "n_still_trading": n_still,
           "median_final_equity": round(float(np.median(final_eqs)), 2), "verdict": verdict}
export_json["monte_carlo_ftmo"] = mc_data
with open(os.path.join(LATEST_DIR, "summary.json"), 'w') as f:
    json.dump(export_json, f, indent=2, default=str)


# ════════════════════════════════════════════════════════════════
# LOG TO GOOGLE SHEETS
# ════════════════════════════════════════════════════════════════
try:
    from lib.sheets_logger import SheetsLogger
    _logger = SheetsLogger()
    _logger.log_result(export_json)
    # Log trades if available
    _trades_path = os.path.join(LATEST_DIR, "trades.csv")
    if os.path.exists(_trades_path):
        _trades_df = pd.read_csv(_trades_path)
        _logger.log_trades(TICKER, STRATEGY_NAME, RUN_ID, _trades_df)
except Exception as _e:
    print(f"⚠️ Sheets logging skipped: {_e}")

# ════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"\U0001f4e6 EXPORT COMPLETE — {STRATEGY_NAME} on {TICKER}")
print(f"{'='*70}")
print(f"  Run ID:       {RUN_ID}")
print(f"  Timestamp:    {RUN_TIMESTAMP.strftime('%B %d, %Y at %I:%M:%S %p')}")
print(f"  Instrument:   {TICKER} ({export_json['metadata']['instrument_type']})")
print(f"  Params:       {param_str}")
print(f"  Sharpe:       {fmt(M['sharpe'],3)} (IS: {fmt(M['is_sharpe'],3)} -> OOS: {fmt(M['oos_sharpe'],3)})")
print(f"  FTMO Verdict: {verdict} ({pass_rate:.1f}% pass rate)")
print(f"{'='*70}")
print(f"\n\U0001f4c2 {STRAT_DIR}/")
print(f"  \u251c\u2500\u2500 latest/")
print(f"  \u2502   \u251c\u2500\u2500 summary.json")
print(f"  \u2502   \u251c\u2500\u2500 daily_returns.csv")
print(f"  \u2502   \u251c\u2500\u2500 trades.csv")
print(f"  \u2502   \u251c\u2500\u2500 grid_results.csv")
print(f"  \u2502   \u2514\u2500\u2500 tearsheet.pdf       \u2190 Share this with Claude!")
print(f"  \u2514\u2500\u2500 archive/")
print(f"      \u251c\u2500\u2500 {RUN_ID}_summary.json")
print(f"      \u2514\u2500\u2500 {RUN_ID}_tearsheet.pdf")
print(f"\n\U0001f4cb run_log.csv ({len(log_combined)} total runs)")
print(f"\n\U0001f4a1 To get my analysis: upload the tearsheet.pdf to our chat.")
print(f"   For deeper analysis: also upload summary.json + daily_returns.csv")

